In [1]:
from pathlib import Path
import pandas as pd
import requests, time, re

# Set paths.
s_path_data = Path(r"C:\2026_06_26_Hackathon\Data")
s_out = s_path_data / "Dallas_311_Code_Concern_CCS_SNIFF"
s_out.mkdir(parents=True, exist_ok=True)

# Set dataset.
dataset_id = "gc4d-8a49"
base_url = f"https://www.dallasopendata.com/resource/{dataset_id}.json"

# Set vague category to investigate.
target_type = "Code Concern - CCS"

# Download Code Concern - CCS rows.
limit = 50000
offset = 0
chunks = []
while True:
    params = {"$where": f"service_request_type = '{target_type}'", "$limit": limit, "$offset": offset}
    print("Downloading offset:", offset)
    r = requests.get(base_url, params=params, timeout=120)
    print("Status:", r.status_code, "URL:", r.url[:200])
    r.raise_for_status()
    data = r.json()
    df_chunk = pd.DataFrame(data)
    print("Rows:", len(df_chunk))
    if len(df_chunk) == 0:
        break
    chunks.append(df_chunk)
    offset += limit
    time.sleep(0.25)

df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
df.to_csv(s_out / "Code_Concern_CCS_RAW.csv", index=False)
print("\nTotal Code Concern rows:", len(df))
print("Columns:", df.columns.to_list())

# Clean date fields.
for col in ["created_date", "update_date", "closed_date", "overall_service_request_due_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "created_date" in df.columns:
    df["created_year"] = df["created_date"].dt.year
    df["created_month"] = df["created_date"].dt.to_period("M").astype(str)
    df["created_day_of_week"] = df["created_date"].dt.day_name()

if "created_date" in df.columns and "closed_date" in df.columns:
    df["days_open"] = ((df["closed_date"] - df["created_date"]).dt.total_seconds() / 86400).round(2)

# Make column summary.
summary_rows = []
for col in df.columns:
    s = df[col].fillna("").astype(str).str.strip()
    summary_rows.append({"column": col, "nonblank_count": int((s != "").sum()), "blank_count": int((s == "").sum()), "unique_count": int(s[s != ""].nunique()), "example_values": " | ".join(s[s != ""].drop_duplicates().head(10).to_list())})
df_summary = pd.DataFrame(summary_rows).sort_values(["nonblank_count", "unique_count"], ascending=[False, False])
df_summary.to_csv(s_out / "SNIFF_column_summary.csv", index=False)

# Save value counts for useful fields.
count_cols = ["status", "outcome", "priority", "method_received_description", "city_council_district", "department", "created_year", "created_month", "created_day_of_week"]
for col in count_cols:
    if col in df.columns:
        temp = df[col].fillna("[NULL]").astype(str).str.strip().replace("", "[BLANK]").value_counts().reset_index()
        temp.columns = [col, "row_count"]
        temp["percent"] = (temp["row_count"] / len(df) * 100).round(2)
        temp.to_csv(s_out / f"COUNT_{col}.csv", index=False)

# Search all text fields for lighting/outage clues.
text_cols = [c for c in df.columns if df[c].dtype == "object"]
df["all_text_search"] = df[text_cols].fillna("").astype(str).agg(" | ".join, axis=1).str.lower()

keyword_patterns = {
    "light_any": r"\blight|\blighting|\bstreetlight|\bstreet light|\blamp|\bdark\b",
    "street_light": r"streetlight|street light|street lamp|light pole|lamp post|pole light",
    "outage_bad_fix": r"outage|out|not working|broken|bad|repair|replace|fixed|dark",
    "oncor": r"oncor",
    "traffic_signal": r"traffic light|traffic signal|signal light",
    "alley_or_parking": r"alley|parking lot|park light|trail light"
}

for label, pattern in keyword_patterns.items():
    df[label] = df["all_text_search"].str.contains(pattern, regex=True, na=False)

df_keyword_counts = pd.DataFrame([{"keyword_group": label, "row_count": int(df[label].sum()), "percent": round(df[label].mean() * 100, 2)} for label in keyword_patterns])
df_keyword_counts.to_csv(s_out / "COUNT_keyword_groups.csv", index=False)

# Pull likely lighting rows from Code Concern.
keyword_cols = list(keyword_patterns.keys())
df_lighting_possible = df[df[keyword_cols].any(axis=1)].copy()
df_lighting_possible.to_csv(s_out / "Code_Concern_CCS_POSSIBLE_LIGHTING_ROWS.csv", index=False)

# Count outcomes only for possible lighting rows.
if len(df_lighting_possible) > 0:
    for col in ["outcome", "status", "priority", "city_council_district", "created_year", "created_month"]:
        if col in df_lighting_possible.columns:
            temp = df_lighting_possible[col].fillna("[NULL]").astype(str).str.strip().replace("", "[BLANK]").value_counts().reset_index()
            temp.columns = [col, "row_count"]
            temp.to_csv(s_out / f"COUNT_POSSIBLE_LIGHTING_{col}.csv", index=False)

# Save a readable sample sorted with light-ish rows first.
sample_cols = [c for c in ["service_request_number", "service_request_type", "status", "created_date", "closed_date", "days_open", "address", "city_council_district", "priority", "method_received_description", "outcome", "unique_key", "lat_location"] if c in df.columns]
df_lighting_possible[sample_cols + keyword_cols].head(1000).to_csv(s_out / "SAMPLE_possible_lighting_rows.csv", index=False)

print("\nSaved folder:", s_out)
print("\nColumn summary:", s_out / "SNIFF_column_summary.csv")
print("All raw Code Concern rows:", s_out / "Code_Concern_CCS_RAW.csv")
print("Keyword counts:", s_out / "COUNT_keyword_groups.csv")
print("Possible lighting rows:", s_out / "Code_Concern_CCS_POSSIBLE_LIGHTING_ROWS.csv")
print("Readable sample:", s_out / "SAMPLE_possible_lighting_rows.csv")

print("\nKeyword counts:")
print(df_keyword_counts.to_string(index=False))

if "outcome" in df.columns:
    print("\nTop outcomes:")
    print(df["outcome"].fillna("[NULL]").astype(str).str.strip().replace("", "[BLANK]").value_counts().head(50).to_string())

if len(df_lighting_possible) > 0:
    print("\nPossible lighting rows:", len(df_lighting_possible))
    print(df_lighting_possible[sample_cols + keyword_cols].head(25).to_string(index=False))
else:
    print("\nNo obvious lighting keywords found inside Code Concern - CCS.")

Status: 200 URL: https://www.dallasopendata.com/resource/gc4d-8a49.json?%24where=service_request_type+%3D+%27Code+Concern+-+CCS%27&%24limit=50000&%24offset=0
Rows: 30649
Status: 200 URL: https://www.dallasopendata.com/resource/gc4d-8a49.json?%24where=service_request_type+%3D+%27Code+Concern+-+CCS%27&%24limit=50000&%24offset=50000
Rows: 0

Total Code Concern rows: 30649
Columns: ['service_request_number', 'address', 'city_council_district', 'department', 'service_request_type', 'ert_estimated_response_time', 'status', 'created_date', 'update_date', 'priority', 'method_received_description', 'unique_key', 'lat_location', 'closed_date', 'outcome', 'overall_service_request_due_date']

Saved folder: C:\2026_06_26_Hackathon\Data\Dallas_311_Code_Concern_CCS_SNIFF

Column summary: C:\2026_06_26_Hackathon\Data\Dallas_311_Code_Concern_CCS_SNIFF\SNIFF_column_summary.csv
All raw Code Concern rows: C:\2026_06_26_Hackathon\Data\Dallas_311_Code_Concern_CCS_SNIFF\Code_Concern_CCS_RAW.csv
Keyword count